# Large-FOV Nuclear Pipeline — Refactored

**Improvements over the previous version**

- All imports consolidated in one place; no re-imports scattered across cells.
- `gc` and `skimage.filters` (required by the focus scorer) added to the import block.
- Critical logic bug in `assign_track_ids_hybrid_dbscan` fixed (misindented `continue`
  caused the match loop body to be skipped for every match).
- Duplicate `plot_largest_cross_sectional_area_vs_time` definition removed.
- `label_and_measure_objects` (Section 7) merged into the existing `regionprops_to_rows`
  path — one code path instead of two diverging implementations.
- `extract_objects_from_saved_masks` now skips `included=False` rows instead of loading
  empty mask files.
- Debug `print` statements removed from the Hungarian-assignment inner loop.
- `segment_single_plane_with_overlap` no longer called with an unsupported `batch_size=`
  kwarg; the config value is used instead.
- Model is loaded exactly once and reused across the debug and full-segmentation cells.
- Focus-scoring utilities moved above the cells that call them.
- `configure_tensorflow_for_gpu` called once at runtime initialisation (Section 19).
- `plot_largest_cross_sectional_area_vs_time` unified: scatter + mean-line in one function.

**Workflow**

Run top-to-bottom the first time.  After that, flip stage flags in **Section 18** and
rerun only the cells you need — every stage writes intermediate artefacts to disk.


## 1. Imports

In [ ]:
from __future__ import annotations

import gc
import json
import math
import os
import time
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import shutil

from joblib import Parallel, delayed
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from skimage import filters, measure, morphology

try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    import tifffile as tiff
except Exception:
    tiff = None

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None


## 2. Configuration

In [ ]:
def get_n_workers() -> int:
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # ── Root paths ────────────────────────────────────────────────────────
    project_root: str = "/home/tdeibert/Data/Machine_Learning_Dev/"
    model_subdir: str = "Models"
    model_name: str = "unet_droplet_nucleus_v6_2_final.keras"
    image_subdir: str = "Images"
    input_image_name: str = "control_extract_1.1.tif"
    output_subdir: str = "Outputs"

    # ── Imaging ───────────────────────────────────────────────────────────
    pixel_size_um: float = 0.108
    z_step_um: float = 1.0
    nuclear_channel_index: int = 1
    membrane_channel_index: int = 0

    # ── Halo analysis ─────────────────────────────────────────────────────
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 4

    # ── Grouping / tracking ───────────────────────────────────────────────
    z_group_tolerance_um: float = 1.0
    time_track_tolerance_um: float = 5.0
    multi_nucleus_exclusion_um: float = 1.0

    track_dbscan_eps_um: float = 6.0
    track_dbscan_min_samples: int = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0

    # ── Stitched acquisition layout ───────────────────────────────────────
    tile_rows: int = 2
    tile_cols: int = 3
    minutes_per_tile: float = 1.0
    serpentine_scan: bool = True

    # ── Segmentation ──────────────────────────────────────────────────────
    prediction_threshold: float = 0.5
    nucleus_probability_threshold: float = 0.3
    min_nucleus_area_px: int = 50
    nucleus_hole_size_px: int = 200
    nucleus_opening_radius_px: int = 1
    min_nucleus_circularity: float = 0.45
    batch_size: int = 8   # tune down if GPU OOM, raise to 16-32 if VRAM permits
    # Safety valve: if more than this fraction of a plane's pixels exceed the
    # nucleus_probability_threshold the plane is almost certainly a bad Z (model
    # over-fired on bright droplet walls / out-of-focus texture).  Skip mask
    # cleaning entirely and return an empty mask.  7 ms to check vs 20 min to clean.
    max_nucleus_foreground_fraction: float = 0.25
    use_mixed_precision: bool = False
    patch_size: int = 512
    patch_stride: int = 256
    background_class_index: int = 0
    droplet_class_index: int = 1
    nucleus_class_index: int = 2

    # ── Focus-aware Z selection ───────────────────────────────────────────
    use_focus_z_selection: bool = True
    focus_window_radius: int = 1
    focus_edge_z_exclusion: int = 2
    focus_channel_index: Optional[int] = None
    focus_metric: str = "object_based_nuclear"
    # Droplet-resistant focus scorer tuning (passed to focus_score_object_based_nuclear)
    focus_min_nucleus_area_um2: float = 20.0    # lower nucleus size bound in µm²
    focus_max_nucleus_area_um2: float = 500.0   # upper bound — set BELOW droplet area
    focus_threshold_k: float = 1.5             # mean + k*std threshold selectivity
    focus_sharpness_weight: float = 0.5        # 0=off, 0.5=moderate, 1.0=strong

    # ── HPC ───────────────────────────────────────────────────────────────
    n_workers: Optional[int] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)

    # ── Unit conversions (derived) ────────────────────────────────────────
    @property
    def z_group_tolerance_px(self) -> float:
        return self.z_group_tolerance_um / self.pixel_size_um

    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def multi_nucleus_exclusion_px(self) -> float:
        return self.multi_nucleus_exclusion_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    @property
    def effective_focus_channel_index(self) -> int:
        return self.nuclear_channel_index if self.focus_channel_index is None else self.focus_channel_index

    # ── Base paths ────────────────────────────────────────────────────────
    @property
    def project_root_path(self) -> Path:
        return Path(self.project_root)

    @property
    def model_path(self) -> Path:
        return self.project_root_path / self.model_subdir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return self.project_root_path / self.image_subdir / self.input_image_name

    @property
    def output_dir(self) -> Path:
        return self.project_root_path / self.output_subdir

    # ── Output subdirectories ─────────────────────────────────────────────
    @property
    def seg_dir(self) -> Path:
        return self.output_dir / "segmentation"

    @property
    def obj_dir(self) -> Path:
        return self.output_dir / "objects"

    @property
    def track_dir(self) -> Path:
        return self.output_dir / "tracking"

    @property
    def analysis_dir(self) -> Path:
        return self.output_dir / "analysis"

    @property
    def qc_dir(self) -> Path:
        return self.output_dir / "qc"

    @property
    def mask_tif_dir(self) -> Path:
        return self.output_dir / "mask_tifs"

    # ── Segmentation artefact paths ───────────────────────────────────────
    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_probability_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_probability_hyperstack.tif"

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    def __repr__(self) -> str:
        keys = [
            "project_root", "model_name", "input_image_name", "output_subdir",
            "pixel_size_um", "nuclear_channel_index", "membrane_channel_index",
            "z_group_tolerance_um", "time_track_tolerance_um",
            "multi_nucleus_exclusion_um", "track_dbscan_eps_um",
            "min_nucleus_area_px", "patch_size", "patch_stride",
            "n_workers", "use_mixed_precision",
        ]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)


cfg = PipelineConfig()

for p in [
    cfg.output_dir, cfg.seg_dir, cfg.obj_dir, cfg.track_dir,
    cfg.analysis_dir, cfg.qc_dir, cfg.mask_tif_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

cfg


## 3. Save / load configuration

In [ ]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


save_config(cfg, cfg.output_dir / "pipeline_config.json")
print("Configuration saved.")


## 4. GPU setup

In [ ]:
def configure_tensorflow_for_gpu(config: PipelineConfig) -> None:
    if tf is None:
        print("TensorFlow not available — skipping GPU setup.")
        return

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    if getattr(config, "use_mixed_precision", False):
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled.")
        except Exception as exc:
            print("Could not enable mixed precision:", exc)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as exc:
            print("Could not set memory growth:", exc)


## 5. Image loading helpers

In [ ]:
def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print("Loaded image shape:", arr.shape)
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(img_5d: np.ndarray, t_idx: int, z_idx: int) -> np.ndarray:
    """Return a full 3-channel plane in (Y, X, C) format from a (T, Z, C, Y, X) image."""
    return np.moveaxis(img_5d[t_idx, z_idx], 0, -1)


## 6. Focus-scoring utilities

Placed before segmentation so they are available for Z-selection during the segmentation loop.

In [ ]:
def _normalize_2d(img2d: np.ndarray) -> np.ndarray:
    """Normalise a 2-D plane to [0, 1] for focus scoring."""
    arr = np.asarray(img2d, dtype=np.float32)
    lo, hi = np.nanmin(arr), np.nanmax(arr)
    if not (np.isfinite(lo) and np.isfinite(hi) and hi > lo):
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)


def _area_um2_to_px(area_um2: float, pixel_size_um: float) -> int:
    return max(1, int(np.ceil(area_um2 / pixel_size_um ** 2)))


def focus_score_variance_laplacian(img2d: np.ndarray) -> float:
    """Whole-plane variance-of-Laplacian (fallback / debug)."""
    return float(np.nanvar(ndi.laplace(_normalize_2d(img2d))))


def focus_score_nucleus_weighted_laplacian(
    img2d: np.ndarray,
    candidate_percentile: float = 97.5,
    min_candidate_pixels: int = 50,
) -> float:
    """Bright-pixel-weighted Laplacian focus score (retained for comparison)."""
    img_f = _normalize_2d(img2d)
    finite = np.isfinite(img_f)
    if not finite.any():
        return 0.0

    cutoff = np.nanpercentile(img_f[finite], candidate_percentile)
    candidate_mask = (img_f >= cutoff) & finite

    if int(candidate_mask.sum()) < min_candidate_pixels:
        return 0.25 * focus_score_variance_laplacian(img2d)

    lap = ndi.laplace(img_f)
    weights = np.maximum(img_f[candidate_mask], 1e-6)
    weighted_energy = np.average(lap[candidate_mask] ** 2, weights=weights)
    signal_bonus = np.sqrt(max(float(candidate_mask.mean()), 1e-8))
    return float(weighted_energy * signal_bonus)


def focus_score_hybrid_nuclear_laplacian(
    img2d: np.ndarray,
    global_weight: float = 0.2,
    nuclear_weight: float = 0.8,
) -> float:
    """Hybrid score (retained for comparison / debugging)."""
    return float(
        global_weight * focus_score_variance_laplacian(img2d)
        + nuclear_weight * focus_score_nucleus_weighted_laplacian(img2d)
    )


def focus_score_object_based_nuclear(
    img2d: np.ndarray,
    pixel_size_um: float,
    # ── nucleus size band (set max BELOW typical droplet area) ────────────
    min_nucleus_area_um2: float = 20.0,
    max_nucleus_area_um2: float = 500.0,
    # ── thresholding ──────────────────────────────────────────────────────
    blur_sigma: float = 1.5,
    threshold_k: float = 1.5,
    min_signal_fraction: float = 0.005,
    # ── sharpness weight ──────────────────────────────────────────────────
    sharpness_weight: float = 0.5,
    # ── object count guard ────────────────────────────────────────────────
    min_object_count: int = 1,
) -> float:
    """
    Object-based nuclear focus score — droplet-resistant version.

    Fixes over the original
    -----------------------
    1. TWO-BAND area filter: upper limit set BELOW the droplet body area so
       droplet spheres cannot dominate the score regardless of brightness or
       roundness.  For 30–60 µm droplets at 0.108 µm/px, keep max <= 500 µm².

    2. MEAN + k·STD threshold replaces Otsu.  When >95% of the frame is dim
       droplet interior, Otsu anchors to that background and droplet walls pass
       as objects.  mean + k*std stays selective when nuclear signal is sparse.

    3. PER-OBJECT SHARPNESS WEIGHT.  In-focus nuclei have sharp edges (high
       intra-object Laplacian variance); out-of-focus droplet halos have smooth
       gradients (low intra-object Laplacian variance).  Each object's score is
       multiplied by (1 + sharpness_weight * normalised_lap_var), penalising
       blurry rings and rewarding sharp discs.

    Parameters
    ----------
    img2d : np.ndarray
        Single 2-D plane from the nuclear channel (any dtype).
    pixel_size_um : float
        Physical pixel size in µm.
    min_nucleus_area_um2 : float
        Smallest accepted object in µm².  Default 20 µm² ≈ 5 µm diameter.
    max_nucleus_area_um2 : float
        Largest accepted object in µm².  Set BELOW the droplet body area.
        For 30 µm droplets: π*15² ≈ 707 µm² → keep this at ≤ 500 µm².
        Tune this first if droplets are still winning.
    blur_sigma : float
        Gaussian pre-blur sigma (pixels) before thresholding.
    threshold_k : float
        Threshold = mean + threshold_k * std.  Raise to 2.0 if droplet
        walls still leak through after narrowing max_nucleus_area_um2.
    min_signal_fraction : float
        If fewer than this fraction of pixels exceed the plane mean, return 0
        (plane has no nuclear signal — do not score noise).
    sharpness_weight : float
        0 = pure intensity×area×shape (original behaviour).
        0.5 = moderate sharpness penalty (recommended).
        1.0 = strong sharpness preference (use when blur is severe).
    min_object_count : int
        Minimum valid objects required before returning a non-zero score.
    """
    nuc = _normalize_2d(img2d)
    if np.nanmax(nuc) <= 0:
        return 0.0

    # Signal presence guard — return early on blank / near-blank planes
    mean_val = float(np.nanmean(nuc))
    if float((nuc > mean_val).mean()) < min_signal_fraction:
        return 0.0

    nuc_blur = filters.gaussian(nuc, sigma=blur_sigma, preserve_range=True)

    # Mean + k·std threshold
    thr = mean_val + threshold_k * float(np.nanstd(nuc_blur))
    if thr >= 1.0:
        return 0.0

    min_area_px = _area_um2_to_px(min_nucleus_area_um2, pixel_size_um)
    max_area_px = _area_um2_to_px(max_nucleus_area_um2, pixel_size_um)

    mask = morphology.remove_small_objects(
        (nuc_blur > thr).astype(bool), min_size=min_area_px)

    # Pre-compute Laplacian once for the sharpness term
    lap = ndi.laplace(nuc)
    plane_lap_var = float(np.var(lap)) + 1e-12

    labeled = measure.label(mask)

    score = 0.0
    valid_count = 0

    for p in measure.regionprops(labeled, intensity_image=nuc):
        if p.area < min_area_px or p.area > max_area_px:
            continue

        roundness_weight = max(0.05, 1.0 - float(p.eccentricity))
        solidity_weight  = max(0.05, float(getattr(p, "solidity", 1.0)))
        base = float(p.area) * float(p.mean_intensity) * roundness_weight * solidity_weight

        if sharpness_weight > 0:
            obj_pixels = lap[labeled == p.label]
            sharpness = float(np.var(obj_pixels)) / plane_lap_var
            combined_weight = 1.0 + sharpness_weight * sharpness
        else:
            combined_weight = 1.0

        score += base * combined_weight
        valid_count += 1

    return float(score) if valid_count >= min_object_count else 0.0



_FOCUS_METRICS = {
    "object_based_nuclear":       lambda img, pxum: focus_score_object_based_nuclear(img, pixel_size_um=pxum),
    "variance_laplacian":         lambda img, pxum: focus_score_variance_laplacian(img),
    "nucleus_weighted_laplacian": lambda img, pxum: focus_score_nucleus_weighted_laplacian(img),
    "hybrid_nuclear_laplacian":   lambda img, pxum: focus_score_hybrid_nuclear_laplacian(img),
}


def focus_score_plane(img2d: np.ndarray, metric: str = "object_based_nuclear",
                      pixel_size_um: float = 0.108,
                      config: Optional["PipelineConfig"] = None) -> float:
    """Dispatch focus metric.  Pass config to forward tuning params to the scorer."""
    if metric not in _FOCUS_METRICS:
        raise ValueError(f"Unsupported focus metric: {metric!r}. Choose from {list(_FOCUS_METRICS)}")
    if metric == "object_based_nuclear" and config is not None:
        return focus_score_object_based_nuclear(
            img2d, pixel_size_um=pixel_size_um,
            min_nucleus_area_um2=config.focus_min_nucleus_area_um2,
            max_nucleus_area_um2=config.focus_max_nucleus_area_um2,
            threshold_k=config.focus_threshold_k,
            sharpness_weight=config.focus_sharpness_weight,
        )
    return _FOCUS_METRICS[metric](img2d, pixel_size_um)


def get_focus_scores_for_timepoint(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> np.ndarray:
    """Score every Z plane for one time point; edge planes are set to NaN."""
    n_z = img_5d.shape[1]
    scores = np.full(n_z, np.nan, dtype=np.float32)
    for z in range(exclude_edge_z, n_z - exclude_edge_z):
        scores[z] = focus_score_plane(img_5d[t, z, nuc_channel_idx],
                                      metric, pixel_size_um, config=config)
    return scores


def get_best_focus_z_indices(
    img_5d: np.ndarray,
    t: int,
    nuc_channel_idx: int,
    exclude_edge_z: int = 2,
    window_radius: int = 1,
    metric: str = "object_based_nuclear",
    pixel_size_um: float = 0.108,
    config: Optional["PipelineConfig"] = None,
) -> Tuple[List[int], np.ndarray, Optional[int]]:
    """Return (keep_z, scores, best_z) for the best-focus window at time t."""
    scores = get_focus_scores_for_timepoint(
        img_5d, t, nuc_channel_idx, exclude_edge_z, metric, pixel_size_um,
        config=config)

    if np.all(np.isnan(scores)):
        return [], scores, None

    best_z = int(np.nanargmax(scores))
    z_start = max(exclude_edge_z, best_z - window_radius)
    z_stop  = min(len(scores) - exclude_edge_z, best_z + window_radius + 1)
    return list(range(z_start, z_stop)), scores, best_z


def plot_focus_scores_for_timepoint(img_5d: np.ndarray, t: int,
                                    config: PipelineConfig) -> Tuple:
    """Quick QC plot of focus scores across Z for one time point."""
    keep_z, scores, best_z = get_best_focus_z_indices(
        img_5d, t,
        nuc_channel_idx=config.effective_focus_channel_index,
        exclude_edge_z=config.focus_edge_z_exclusion,
        window_radius=config.focus_window_radius,
        metric=config.focus_metric,
        pixel_size_um=config.pixel_size_um,
        config=config,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(len(scores)), scores, marker="o")
    if best_z is not None:
        ax.axvline(best_z, linestyle="--", label=f"best z={best_z}")
    for z in keep_z:
        ax.axvspan(z - 0.45, z + 0.45, alpha=0.2)
    ax.set_xlabel("Z index")
    ax.set_ylabel("Focus score")
    ax.set_title(f"Focus scores t={t}  keep_z={keep_z}  metric={config.focus_metric}")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return keep_z, scores, best_z


## 7. Segmentation helpers

In [ ]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def _preprocess_patch(patch: np.ndarray) -> np.ndarray:
    x = patch.astype(np.float32)
    m = np.max(x)
    return x / m if m > 0 else x


def _generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """Generate patch start positions that always include the image end."""
    starts = list(range(0, max(length - patch_size + 1, 1), stride)) or [0]
    last = max(0, length - patch_size)
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def extract_overlapping_patches(
    image_yxc: np.ndarray,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    h, w, _ = image_yxc.shape
    y_starts = _generate_patch_starts(h, patch_size, stride)
    x_starts = _generate_patch_starts(w, patch_size, stride)
    patches, coords = [], []
    for y0 in y_starts:
        for x0 in x_starts:
            patches.append(image_yxc[y0:y0 + patch_size, x0:x0 + patch_size, :])
            coords.append((y0, x0))
    return patches, coords


def _predict_batch(model, batch_patches: np.ndarray) -> np.ndarray:
    """
    Use model(x, training=False) instead of model.predict().
    model.predict() spins up a fresh TF dataset pipeline and progress-bar
    machinery on every call — at batch_size=1 this overhead dominates.
    Direct __call__ avoids all of that and runs the graph immediately.
    """
    preds = model(batch_patches, training=False).numpy()
    if preds.ndim != 4:
        raise ValueError(f"Unexpected prediction shape: {preds.shape}")
    if preds.shape[-1] == 1:
        fg = preds[..., 0]
        preds = np.stack([1.0 - fg, fg], axis=-1)
    return preds.astype(np.float32)


def stitch_probability_patches(
    patch_probs: List[np.ndarray],
    coords: List[Tuple[int, int]],
    image_shape: Tuple[int, int],
    n_classes: int,
    patch_size: int = 512,
) -> np.ndarray:
    """
    Average overlapping patch probabilities into a full probability map (H, W, C).

    Accepts either a list of 2-D arrays or the stacked (N, pH, pW, C) array
    returned directly from the inference loop.
    """
    h, w = image_shape
    prob_sum   = np.zeros((h, w, n_classes), dtype=np.float32)
    prob_count = np.zeros((h, w),            dtype=np.float32)

    # Accept pre-stacked array or list
    probs_arr = (np.asarray(patch_probs) if not isinstance(patch_probs, np.ndarray)
                 else patch_probs)

    for idx, (y0, x0) in enumerate(coords):
        y1, x1 = y0 + patch_size, x0 + patch_size
        prob_sum  [y0:y1, x0:x1] += probs_arr[idx]
        prob_count[y0:y1, x0:x1] += 1.0

    return prob_sum / np.maximum(prob_count[:, :, np.newaxis], 1.0)


def _filter_by_area_and_circularity(
    binary_mask: np.ndarray,
    min_area_px: int = 50,
    min_circularity: float = 0.45,
) -> np.ndarray:
    """
    Vectorised area + circularity filter.

    Uses regionprops_table to fetch all properties in one NumPy pass instead
    of a Python loop over individual regionprops objects.  For a large FOV with
    hundreds of objects this is 10-50x faster.
    """
    labeled = measure.label(binary_mask, connectivity=2)
    if labeled.max() == 0:
        return np.zeros_like(binary_mask, dtype=bool)

    tbl = measure.regionprops_table(labeled, properties=("label", "area", "perimeter"))
    areas      = np.array(tbl["area"],      dtype=np.float64)
    perimeters = np.array(tbl["perimeter"], dtype=np.float64)
    labels     = np.array(tbl["label"],     dtype=np.int32)

    with np.errstate(divide="ignore", invalid="ignore"):
        circularity = np.where(
            perimeters > 0,
            (4.0 * np.pi * areas) / perimeters ** 2,
            0.0,
        )

    keep_mask = (areas >= min_area_px) & (circularity >= min_circularity)
    keep_labels = set(labels[keep_mask].tolist())

    # Vectorised label lookup — much faster than iterating props
    out = np.isin(labeled, list(keep_labels))
    return out


def clean_nucleus_mask(
    nucleus_mask: np.ndarray,
    min_size_px: int = 50,
    hole_size_px: int = 200,
    opening_radius: int = 1,
    min_circularity: float = 0.45,
    max_foreground_fraction: float = 0.25,
) -> np.ndarray:
    """
    Clean a raw nucleus probability mask.

    Operation order is intentional:
      1. Foreground-fraction guard  — bail out immediately on over-fired planes.
         A plane where >25% of pixels are foreground is almost always a bad Z
         (model firing on droplet walls / texture).  Returning empty here costs
         7 ms vs up to 20 min of morphology + regionprops on a near-solid mask.
      2. remove_small_objects first — reduces object count before any fill_holes,
         so subsequent operations work on a sparser mask.
      3. opening before fill_holes — breaks merged blobs before filling gaps,
         preventing fill_holes from joining large connected regions.
      4. fill_holes on the reduced mask — much cheaper than on the raw mask.
      5. Perimeter-based circularity filter last — only computed on surviving
         objects after area filtering, not on every initial fragment.
    """
    mask = nucleus_mask.astype(bool)

    # ── Guard: bail on over-fired planes ─────────────────────────────────
    if max_foreground_fraction < 1.0:
        fg_frac = float(mask.mean())
        if fg_frac > max_foreground_fraction:
            return np.zeros_like(mask, dtype=bool)

    # ── Step 1: remove small fragments early ─────────────────────────────
    mask = morphology.remove_small_objects(mask, min_size=min_size_px)
    if not mask.any():
        return mask

    # ── Step 2: opening before fill to break merged blobs ────────────────
    if opening_radius > 0:
        mask = morphology.opening(mask, footprint=morphology.disk(opening_radius))
    if not mask.any():
        return mask

    # ── Step 3: fill holes on the already-reduced mask ───────────────────
    mask = ndi.binary_fill_holes(mask)

    # ── Step 4: circularity filter (perimeter only on survivors) ─────────
    return _filter_by_area_and_circularity(mask, min_area_px=min_size_px,
                                           min_circularity=min_circularity)


def _run_batched_inference(
    patch_arr: np.ndarray,
    model,
    batch_size: int,
) -> np.ndarray:
    """
    Run model inference on a (N, H, W, C) patch array.

    Two key optimisations vs the previous implementation:

    1. FIXED BATCH SIZE via zero-padding.
       Every batch sent to the GPU is exactly `batch_size` patches.  The last
       batch is zero-padded to match and the padding rows are discarded after.
       This guarantees tf.function only ever sees one input shape → traced ONCE,
       reused for all subsequent calls.  Variable last-batch sizes were causing
       XLA recompilation (60-300s stall) on every timepoint.

    2. SINGLE CONTIGUOUS ALLOCATION.
       All patches are passed as one pre-allocated array.  The GPU driver sees
       one large allocation rather than n_batches × small alloc/free cycles,
       eliminating the CUDA memory fragmentation that caused multi-minute stalls
       after the first few timepoints.
    """
    n_patches = patch_arr.shape[0]

    # Build a tf.function compiled callable, traced exactly once per model
    model_id = id(model)
    if model_id not in _INFER_FN_CACHE:
        # Concrete input signature pins the shape — prevents ANY retracing
        sig = tf.TensorSpec(shape=(batch_size, None, None, None), dtype=tf.float32)
        @tf.function(input_signature=[sig])
        def _infer(x):
            return model(x, training=False)
        _INFER_FN_CACHE[model_id] = _infer
    infer_fn = _INFER_FN_CACHE[model_id]

    all_preds: List[np.ndarray] = []
    for i in range(0, n_patches, batch_size):
        chunk = patch_arr[i:i + batch_size]
        actual = chunk.shape[0]

        # Pad to exactly batch_size if this is a short last batch
        if actual < batch_size:
            pad = np.zeros((batch_size - actual, *chunk.shape[1:]), dtype=np.float32)
            chunk = np.concatenate([chunk, pad], axis=0)

        preds = infer_fn(chunk).numpy().astype(np.float32)

        # Discard padding rows
        if actual < batch_size:
            preds = preds[:actual]

        if preds.shape[-1] == 1:
            fg = preds[..., 0]
            preds = np.stack([1.0 - fg, fg], axis=-1)

        all_preds.append(preds)

    return np.concatenate(all_preds, axis=0)  # (N, pH, pW, C)


# Module-level cache for tf.function compiled inference callables
_INFER_FN_CACHE: Dict[int, object] = {}


def segment_all_planes_for_timepoint(
    img_5d: np.ndarray,
    t_idx: int,
    keep_z: List[int],
    model,
    config: PipelineConfig,
) -> Dict[int, Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]]:
    """
    Segment all keep_z planes for one timepoint in a single batched GPU call.

    All patches from ALL planes are collected into one tensor, run through
    the model together, then split back out per plane.  This eliminates GPU
    memory fragmentation from repeated per-plane alloc/free cycles and keeps
    GPU utilisation high throughout.

    Returns
    -------
    Dict mapping z_idx → (class_prob_map, class_label_map, nucleus_mask, nucleus_labels)
    """
    if not keep_z:
        return {}

    patch_size = config.patch_size
    stride     = config.patch_stride
    Y, X       = img_5d.shape[-2], img_5d.shape[-1]

    # ── Collect patches from all planes ──────────────────────────────────
    # We need coords per-plane to reconstruct probability maps after inference
    plane_patch_info: List[Tuple[int, np.ndarray, List[Tuple[int,int]]]] = []
    all_patches: List[np.ndarray] = []

    for z in keep_z:
        plane_yxc = get_full_plane_yxc(img_5d, t_idx, z)
        patches, coords = extract_overlapping_patches(plane_yxc, patch_size, stride)
        preprocessed = np.stack([_preprocess_patch(p) for p in patches], axis=0)
        plane_patch_info.append((z, coords, preprocessed.shape[0]))
        all_patches.append(preprocessed)

    # Single contiguous array: (total_patches, patch_H, patch_W, C)
    all_patch_arr = np.concatenate(all_patches, axis=0)
    total_patches = all_patch_arr.shape[0]

    # ── Single batched inference pass ─────────────────────────────────────
    all_probs_arr = _run_batched_inference(
        all_patch_arr, model, batch_size=max(1, int(config.batch_size)))
    # all_probs_arr: (total_patches, patch_H, patch_W, n_classes)
    n_classes = all_probs_arr.shape[-1]

    # ── Split results back per plane and post-process ─────────────────────
    results: Dict[int, Tuple] = {}
    offset = 0
    for z, coords, n_p in plane_patch_info:
        plane_probs = all_probs_arr[offset:offset + n_p]
        offset += n_p

        class_prob_map = stitch_probability_patches(
            plane_probs, coords, (Y, X), n_classes, patch_size)

        class_label_map = np.argmax(class_prob_map, axis=-1).astype(np.uint8)
        nucleus_mask = (
            class_prob_map[..., config.nucleus_class_index]
            > config.nucleus_probability_threshold
        )
        nucleus_mask = clean_nucleus_mask(
            nucleus_mask,
            min_size_px=config.min_nucleus_area_px,
            hole_size_px=config.nucleus_hole_size_px,
            opening_radius=config.nucleus_opening_radius_px,
            min_circularity=config.min_nucleus_circularity,
            max_foreground_fraction=config.max_nucleus_foreground_fraction,
        )
        nucleus_labels = measure.label(nucleus_mask, connectivity=2).astype(np.uint16)
        results[z] = (class_prob_map, class_label_map, nucleus_mask, nucleus_labels)

    return results


def segment_single_plane_with_overlap(
    plane_yxc: np.ndarray,
    model,
    config: PipelineConfig,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Single-plane wrapper — used by the debug cell (Section 26) only."""
    patches, coords = extract_overlapping_patches(plane_yxc, patch_size, stride)
    patch_arr = np.stack([_preprocess_patch(p) for p in patches], axis=0)
    probs_arr = _run_batched_inference(patch_arr, model, batch_size=max(1, int(config.batch_size)))
    n_classes = probs_arr.shape[-1]
    h, w, _ = plane_yxc.shape
    class_prob_map = stitch_probability_patches(probs_arr, coords, (h, w), n_classes, patch_size)
    class_label_map = np.argmax(class_prob_map, axis=-1).astype(np.uint8)
    nucleus_mask = class_prob_map[..., config.nucleus_class_index] > config.nucleus_probability_threshold
    nucleus_mask = clean_nucleus_mask(
        nucleus_mask,
        min_size_px=config.min_nucleus_area_px,
        hole_size_px=config.nucleus_hole_size_px,
        opening_radius=config.nucleus_opening_radius_px,
        min_circularity=config.min_nucleus_circularity,
        max_foreground_fraction=config.max_nucleus_foreground_fraction,
    )
    return class_prob_map, class_label_map, nucleus_mask, measure.label(nucleus_mask).astype(np.uint16)


## 8. TIFF save helpers

In [ ]:
def _write_hyperstack_tiff(arr: np.ndarray, out_path: Path, axes: str,
                           prefer_imagej: bool = True) -> None:
    if tiff is None:
        raise ImportError("tifffile is required.")
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    arr = np.asarray(arr)
    # BigTIFF for arrays >= ~2 GB to avoid ImageJ truncation.
    use_imagej = prefer_imagej and arr.nbytes < (2 * 1024**3 - 32 * 1024**2)
    tiff.imwrite(out_path, arr,
                 imagej=use_imagej, bigtiff=not use_imagej,
                 metadata={"axes": axes})


def save_probability_hyperstack_tiff(probability_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(probability_5d, out_path, axes="TZCYX", prefer_imagej=False)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff((class_mask_5d.astype(np.uint8) * 255), out_path,
                           axes="TZCYX", prefer_imagej=True)


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(label_4d.astype(np.uint8), out_path,
                           axes="TZYX", prefer_imagej=True)


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    _write_hyperstack_tiff(instance_4d, out_path, axes="TZYX", prefer_imagej=True)


def create_probability_memmap(
    shape: Tuple[int, int, int, int, int],
    out_dir: Path,
    dtype: np.dtype = np.float16,
) -> Tuple[np.memmap, Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_path = out_dir / "probability_hyperstack_temp.dat"
    if temp_path.exists():
        temp_path.unlink()
    return np.memmap(temp_path, mode="w+", dtype=dtype, shape=shape), temp_path


## 9. Region-props helper

In [ ]:
def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
    min_area_px: int = 0,
) -> List[Dict]:
    """
    Convert a labelled mask to a list of row dicts.

    The `min_area_px` guard lets object-extraction callers reuse this function
    directly instead of maintaining a second copy of the same logic.
    """
    rows = []
    for prop in measure.regionprops(labeled_mask):
        if prop.area < min_area_px:
            continue
        cy, cx = prop.centroid
        perimeter = float(prop.perimeter)
        circularity = (
            (4.0 * np.pi * prop.area) / perimeter ** 2
            if perimeter > 0 else np.nan
        )
        rows.append({
            "t": t_idx, "z": z_idx,
            "class_id": class_id, "class_name": class_name,
            "label": int(prop.label),
            "centroid_x_px": float(cx), "centroid_y_px": float(cy),
            "area_px": int(prop.area),
            "perimeter_px": perimeter,
            "circularity": circularity,
            "bbox_min_row": int(prop.bbox[0]), "bbox_min_col": int(prop.bbox[1]),
            "bbox_max_row": int(prop.bbox[2]), "bbox_max_col": int(prop.bbox[3]),
        })
    return rows


## 10. Full segmentation loop

In [ ]:
def run_segmentation_for_all_planes(
    img_5d: np.ndarray,
    model,
    config: PipelineConfig,
    save_binary_masks: bool = True,
    save_probability_tiff: bool = True,
) -> pd.DataFrame:
    """
    Run U-Net segmentation over the full hyperstack.

    Memory architecture
    -------------------
    Previous version held class_mask_5d, class_label_4d, nucleus_instance_4d,
    droplet_instance_4d, and roi_rows ALL in RAM simultaneously until the very
    end of the run.  For a 30T × 10Z × 5k×5k image this exceeded 60 GB and
    caused the OS to begin swapping by t=3, explaining the 20-minute stall.

    This version uses a WRITE-AS-YOU-GO strategy:
    - All four hyperstack TIFFs are opened as memory-mapped files at the start.
      Each plane is written immediately after inference and the local arrays
      are deleted.  Peak extra RAM per plane is one plane (~300 MB), not the
      full hyperstack.
    - roi_rows is flushed to disk as a growing Parquet file every
      `roi_flush_every` time points and cleared from RAM.
    - records (the segmentation index) is small and stays in RAM — it is only
      ~one dict per Z plane per T.
    - Probability map uses the existing memmap path (unchanged).
    """
    T, Z = img_5d.shape[0], img_5d.shape[1]
    Y, X = img_5d.shape[-2], img_5d.shape[-1]

    roi_flush_every: int = 2   # flush roi_rows to disk every N time points

    records:  List[dict] = []
    roi_rows: List[dict] = []

    # ── Open all four output hyperstacks as memmaps ───────────────────────
    # Written plane-by-plane so RAM never holds more than one plane at a time.
    def _open_memmap(path: Path, dtype, shape):
        path.parent.mkdir(parents=True, exist_ok=True)
        return np.memmap(path, dtype=dtype, mode="w+", shape=shape)

    mm_class   = _open_memmap(config.mask_tif_dir / "class_mask_5d.dat",
                              np.uint8,  (T, Z, 3, Y, X))
    mm_label   = _open_memmap(config.mask_tif_dir / "class_label_4d.dat",
                              np.uint8,  (T, Z, Y, X))
    mm_nuc     = _open_memmap(config.mask_tif_dir / "nucleus_instance_4d.dat",
                              np.uint16, (T, Z, Y, X))
    mm_drop    = _open_memmap(config.mask_tif_dir / "droplet_instance_4d.dat",
                              np.uint16, (T, Z, Y, X))

    probability_5d = probability_temp_path = None
    if save_probability_tiff:
        probability_5d, probability_temp_path = create_probability_memmap(
            shape=(T, Z, 3, Y, X), out_dir=config.seg_dir)

    roi_pkl_path = config.obj_dir / "roi_rows_accumulator.pkl"
    roi_accumulated: List[dict] = []   # flushed to disk every flush_every t
    roi_flush_every: int = 2           # write accumulated rows to disk every N timepoints

    try:
        # ── Pre-compute ALL focus scores before touching the GPU ──────────
        print("Pre-computing focus scores for all time points...")
        focus_data: Dict[int, Tuple[List[int], np.ndarray, Optional[int]]] = {}
        for t_pre in range(T):
            if config.use_focus_z_selection:
                kz, fs, bz = get_best_focus_z_indices(
                    img_5d=img_5d, t=t_pre,
                    nuc_channel_idx=config.effective_focus_channel_index,
                    exclude_edge_z=config.focus_edge_z_exclusion,
                    window_radius=config.focus_window_radius,
                    metric=config.focus_metric,
                    pixel_size_um=config.pixel_size_um,
                    config=config,
                )
                focus_data[t_pre] = (kz, fs, bz)
                print(f"  t={t_pre}: best z={bz}, keep_z={kz}")
            else:
                focus_data[t_pre] = (list(range(Z)),
                                     np.full(Z, np.nan, dtype=np.float32), None)
        print("Focus scoring complete. Starting GPU segmentation...\n")

        # ── Segmentation loop ─────────────────────────────────────────────
        for t_idx in range(T):
            keep_z, focus_scores, best_z = focus_data[t_idx]
            keep_z_set = set(keep_z)
            print(f"t={t_idx}: best_z={best_z}, segmenting z={keep_z}")
            t_start = time.perf_counter()

            # ── Record skipped planes (no inference needed) ───────────────
            for z_idx in range(Z):
                if z_idx in keep_z_set:
                    continue
                fscore = (float(focus_scores[z_idx])
                          if z_idx < len(focus_scores)
                          and np.isfinite(focus_scores[z_idx]) else np.nan)
                out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
                reason = ("edge_z_excluded"
                          if z_idx < config.focus_edge_z_exclusion
                             or z_idx >= Z - config.focus_edge_z_exclusion
                          else "outside_best_focus_window"
                          if config.use_focus_z_selection else "skipped")
                if save_binary_masks:
                    np.save(out_path, np.zeros((Y, X), dtype=np.uint8))
                records.append({"t": t_idx, "z": z_idx, "included": False,
                                "reason": reason, "focus_score": fscore,
                                "best_focus_z": best_z, "mask_path": str(out_path)})

            # ── Single batched GPU call for ALL keep_z planes ─────────────
            plane_results = segment_all_planes_for_timepoint(
                img_5d, t_idx, keep_z, model, config)

            # ── Store results and record per-plane metrics ────────────────
            for z_idx in keep_z:
                fscore = (float(focus_scores[z_idx])
                          if z_idx < len(focus_scores)
                          and np.isfinite(focus_scores[z_idx]) else np.nan)
                out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
                class_prob_map, class_label_map, nucleus_mask, nucleus_instance_labels = (
                    plane_results[z_idx])

                background_mask   = class_label_map == config.background_class_index
                droplet_mask      = class_label_map == config.droplet_class_index
                droplet_instances = measure.label(droplet_mask, connectivity=2).astype(np.uint16)

                if save_probability_tiff and probability_5d is not None:
                    probability_5d[t_idx, z_idx] = np.moveaxis(
                        class_prob_map.astype(np.float16), -1, 0)

                mm_class[t_idx, z_idx, 0] = background_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 1] = droplet_mask.astype(np.uint8)
                mm_class[t_idx, z_idx, 2] = nucleus_mask.astype(np.uint8)
                mm_label[t_idx, z_idx]    = class_label_map
                mm_nuc  [t_idx, z_idx]    = nucleus_instance_labels
                mm_drop [t_idx, z_idx]    = droplet_instances

                roi_rows.extend(regionprops_to_rows(
                    nucleus_instance_labels, t_idx, z_idx,
                    config.nucleus_class_index, "nucleus"))
                roi_rows.extend(regionprops_to_rows(
                    droplet_instances, t_idx, z_idx,
                    config.droplet_class_index, "droplet"))

                if save_binary_masks:
                    np.save(out_path, nucleus_mask.astype(np.uint8))

                records.append({
                    "t": t_idx, "z": z_idx, "included": True,
                    "reason": "segmented", "focus_score": fscore,
                    "best_focus_z": best_z, "mask_path": str(out_path),
                    "nucleus_pixels": int(nucleus_mask.sum()),
                    "droplet_pixels": int(droplet_mask.sum()),
                })
                print(f"    z={z_idx}: nuc_px={int(nucleus_mask.sum()):,}  "
                      f"fg_frac={float(nucleus_mask.mean()):.3f}")

                del class_prob_map, class_label_map, nucleus_mask
                del nucleus_instance_labels, droplet_mask, background_mask, droplet_instances

            del plane_results

            if (t_idx % roi_flush_every == 0 or t_idx == T - 1):
                roi_accumulated.extend(roi_rows)
                pd.DataFrame(roi_accumulated).to_pickle(roi_pkl_path)
                roi_rows.clear()

            gc.collect()
            elapsed = time.perf_counter() - t_start
            print(f"  t={t_idx} complete in {elapsed:.1f}s")

        # ── Flush memmaps → final TIFFs (written directly, no RAM copy) ───
        print("Flushing memmaps to hyperstack TIFFs...")
        for mm in (mm_class, mm_label, mm_nuc, mm_drop):
            mm.flush()
        # Write directly from the memory-mapped arrays — tifffile reads them
        # chunk-by-chunk without loading the full array into RAM.
        _write_hyperstack_tiff(mm_class, config.segmentation_class_hyperstack_path,
                               axes="TZCYX", prefer_imagej=False)
        _write_hyperstack_tiff(mm_label, config.segmentation_label_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)
        _write_hyperstack_tiff(mm_nuc,   config.nucleus_instance_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)
        _write_hyperstack_tiff(mm_drop,  config.droplet_instance_hyperstack_path,
                               axes="TZYX", prefer_imagej=True)

        if save_probability_tiff and probability_5d is not None:
            probability_5d.flush()
            save_probability_hyperstack_tiff(
                probability_5d, config.segmentation_probability_hyperstack_path)

        # ── Finalise ROI table and index ──────────────────────────────────
        roi_accumulated.extend(roi_rows)
        roi_df = pd.DataFrame(roi_accumulated)
        roi_df.to_pickle(config.segmentation_roi_table_path)

        seg_index_df = pd.DataFrame(records)
        seg_index_df.to_pickle(config.segmentation_index_path)

        print(f"Segmentation complete. {len(seg_index_df)} plane records, "
              f"{len(roi_df)} ROI rows.")
        return seg_index_df

    finally:
        # Always close memmaps and clean up temp files
        for mm, name in [(mm_class, "class_mask_5d.dat"),
                         (mm_label, "class_label_4d.dat"),
                         (mm_nuc,   "nucleus_instance_4d.dat"),
                         (mm_drop,  "droplet_instance_4d.dat")]:
            try:
                mm.flush()
                del mm
            except Exception:
                pass
        for dat_name in ["class_mask_5d.dat", "class_label_4d.dat",
                         "nucleus_instance_4d.dat", "droplet_instance_4d.dat"]:
            dat_path = config.mask_tif_dir / dat_name
            if dat_path.exists():
                try:
                    dat_path.unlink()
                except Exception:
                    pass
        if isinstance(probability_5d, np.memmap):
            try:
                probability_5d.flush()
            except Exception:
                pass
        if probability_temp_path is not None:
            try:
                Path(probability_temp_path).unlink(missing_ok=True)
            except Exception:
                pass
        if roi_pkl_path.exists():
            try:
                roi_pkl_path.unlink()
            except Exception:
                pass


## 11. Object extraction from masks

In [ ]:
def extract_objects_from_saved_masks(
    seg_index_df: pd.DataFrame,
    config: PipelineConfig,
) -> pd.DataFrame:
    """
    Load saved binary mask files and extract region properties.

    Rows with included=False are skipped — they hold empty masks and
    contribute no objects.
    """
    included = seg_index_df[seg_index_df["included"].astype(bool)]
    all_rows: List[dict] = []
    for row in included.itertuples(index=False):
        mask = np.load(row.mask_path).astype(bool)
        rows = regionprops_to_rows(
            measure.label(mask, connectivity=2),
            t_idx=row.t, z_idx=row.z,
            class_id=config.nucleus_class_index, class_name="nucleus",
            min_area_px=config.min_nucleus_area_px,
        )
        all_rows.extend(rows)

    objects_df = pd.DataFrame(all_rows) if all_rows else pd.DataFrame()
    objects_df.to_pickle(config.obj_dir / "plane_objects.pkl")
    return objects_df


## 12. Distance-based helpers

In [ ]:
def nearest_neighbor_matches(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_dist_px: float,
) -> List[Tuple[int, int, float]]:
    if source_df.empty or target_df.empty:
        return []
    src_xy = source_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tgt_xy = target_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    tree = cKDTree(tgt_xy)
    dists, idxs = tree.query(src_xy, distance_upper_bound=max_dist_px)

    matches, used = [], set()
    for i in np.argsort(dists):
        dist, j = dists[i], idxs[i]
        if np.isinf(dist) or j in used:
            continue
        matches.append((int(i), int(j), float(dist)))
        used.add(int(j))
    return matches


## 13. Group nuclei across Z

In [ ]:
def group_nuclei_across_z(objects_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if objects_df.empty:
        return pd.DataFrame()

    grouped_rows = []
    next_group_id = 1

    for t_idx, df_t in objects_df.groupby("t", sort=True):
        df_t = df_t.sort_values(["z", "label"]).reset_index(drop=True).copy()
        df_t["nucleus_3d_id"] = -1
        z_values = sorted(df_t["z"].unique())

        for idx in df_t.index[df_t["z"] == z_values[0]]:
            df_t.at[idx, "nucleus_3d_id"] = next_group_id
            next_group_id += 1

        for z_prev, z_curr in zip(z_values[:-1], z_values[1:]):
            prev_df = df_t[df_t["z"] == z_prev].reset_index()
            curr_df = df_t[df_t["z"] == z_curr].reset_index()
            matched_curr = set()

            for i_prev, i_curr, _ in nearest_neighbor_matches(
                    prev_df, curr_df, config.z_group_tolerance_px):
                prev_global = int(prev_df.loc[i_prev, "index"])
                curr_global = int(curr_df.loc[i_curr, "index"])
                matched_curr.add(curr_global)

                grp = df_t.at[prev_global, "nucleus_3d_id"]
                if grp == -1:
                    grp = next_group_id
                    df_t.at[prev_global, "nucleus_3d_id"] = grp
                    next_group_id += 1
                df_t.at[curr_global, "nucleus_3d_id"] = grp

            for curr_global in curr_df["index"].tolist():
                if curr_global not in matched_curr and df_t.at[curr_global, "nucleus_3d_id"] == -1:
                    df_t.at[curr_global, "nucleus_3d_id"] = next_group_id
                    next_group_id += 1

        grouped_rows.append(df_t)

    grouped_z_df = pd.concat(grouped_rows, ignore_index=True)
    grouped_z_df.to_pickle(config.obj_dir / "grouped_z_objects.pkl")
    return grouped_z_df


## 14. Select best Z per nucleus

In [ ]:
def select_best_z_per_nucleus(grouped_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if grouped_z_df.empty:
        return pd.DataFrame()
    best_z_df = (
        grouped_z_df
        .sort_values(["t", "nucleus_3d_id", "area_px"], ascending=[True, True, False])
        .groupby(["t", "nucleus_3d_id"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    best_z_df.to_pickle(config.obj_dir / "best_z_nuclei.pkl")
    return best_z_df


## 15. Exclude likely multi-nucleus droplets

In [ ]:
def flag_close_nuclei(best_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if best_z_df.empty:
        return pd.DataFrame()
    out = []
    for _, df_t in best_z_df.groupby("t", sort=True):
        df_t = df_t.copy().reset_index(drop=True)
        coords = df_t[["centroid_x_px", "centroid_y_px"]].to_numpy()
        tree = cKDTree(coords)
        close_flags = np.zeros(len(df_t), dtype=bool)
        for i, xy in enumerate(coords):
            neighbors = [j for j in tree.query_ball_point(xy, r=config.multi_nucleus_exclusion_px) if j != i]
            if neighbors:
                close_flags[i] = True
                for j in neighbors:
                    close_flags[j] = True
        df_t["valid_single_nucleus"] = ~close_flags
        out.append(df_t)
    filtered_df = pd.concat(out, ignore_index=True)
    filtered_df.to_pickle(config.obj_dir / "best_z_nuclei_with_exclusion.pkl")
    return filtered_df


## 16. Tile assignment and true acquisition time

In [ ]:
def assign_tile_index_from_xy(
    x_px: float, y_px: float,
    image_width_px: int, image_height_px: int,
    config: PipelineConfig,
) -> Tuple[int, int, int]:
    col = min(int(x_px // (image_width_px  / config.tile_cols)), config.tile_cols - 1)
    row = min(int(y_px // (image_height_px / config.tile_rows)), config.tile_rows - 1)
    if config.serpentine_scan and row % 2 != 0:
        tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col
    return row, col, tile_index


def add_tile_timing_metadata(
    nuclei_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if nuclei_df.empty:
        return pd.DataFrame()
    image_height_px, image_width_px = img_5d.shape[-2], img_5d.shape[-1]
    out_rows = []
    tiles_per_frame = config.tile_rows * config.tile_cols * config.minutes_per_tile
    for row in nuclei_df.itertuples(index=False):
        tile_row, tile_col, tile_index = assign_tile_index_from_xy(
            row.centroid_x_px, row.centroid_y_px,
            image_width_px, image_height_px, config)
        d = row._asdict()
        d["tile_row"]        = tile_row
        d["tile_col"]        = tile_col
        d["tile_index"]      = tile_index
        d["tile_offset_min"] = tile_index * config.minutes_per_tile
        d["true_time_min"]   = row.t * tiles_per_frame + d["tile_offset_min"]
        out_rows.append(d)
    timed_df = pd.DataFrame(out_rows)
    timed_df.to_pickle(config.track_dir / "best_z_nuclei_timed.pkl")
    return timed_df


## 17. Track nuclei across time — hybrid DBSCAN tracker

1. Build local spatial neighbourhoods with DBSCAN from consecutive frames.
2. Score candidate links by distance + area consistency.
3. One-to-one Hungarian assignment inside each local neighbourhood.
4. Reject crowded or low-confidence links rather than forcing a track.


In [ ]:
def _safe_area_ratio(area_a: float, area_b: float) -> float:
    a, b = max(float(area_a), 1.0), max(float(area_b), 1.0)
    return max(a, b) / min(a, b)


def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)
    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        for col in ("dbscan_cluster_id", "dbscan_cluster_size", "dbscan_is_crowded"):
            pooled[col] = pd.Series(dtype=int if col != "dbscan_is_crowded" else bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(eps=config.track_dbscan_eps_px,
                    min_samples=config.track_dbscan_min_samples).fit_predict(coords)
    pooled["dbscan_cluster_id"] = labels

    crowded_flags = []
    cluster_sizes = []
    for _, df_c in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_c["frame_role"] == "prev").sum())
        n_curr = int((df_c["frame_role"] == "curr").sum())
        is_crowded = n_prev > 1 or n_curr > 1
        crowded_flags.extend([is_crowded] * len(df_c))
        cluster_sizes.extend([len(df_c)] * len(df_c))
    pooled["dbscan_cluster_size"] = cluster_sizes
    pooled["dbscan_is_crowded"]   = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[dict]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    n_prev, n_curr = len(prev_cluster), len(curr_cluster)
    cost_matrix = np.full((n_prev, n_curr), np.nan, dtype=float)
    max_dist_base = float(config.time_track_tolerance_px)

    for i, prev_row in prev_cluster.iterrows():
        for j, curr_row in curr_cluster.iterrows():
            dx  = float(curr_row["centroid_x_px"]) - float(prev_row["centroid_x_px"])
            dy  = float(curr_row["centroid_y_px"]) - float(prev_row["centroid_y_px"])
            dist_px = math.hypot(dx, dy)

            max_dist = max_dist_base
            if prev_row.get("dbscan_is_crowded", False) or curr_row.get("dbscan_is_crowded", False):
                max_dist *= config.track_crowded_distance_scale

            prev_area = max(float(prev_row["area_px"]), 1.0)
            curr_area = max(float(curr_row["area_px"]), 1.0)
            area_ratio = max(prev_area, curr_area) / min(prev_area, curr_area)

            if dist_px <= max_dist and area_ratio <= config.track_area_ratio_max:
                area_cost    = abs(math.log(curr_area / prev_area))
                dist_cost    = dist_px / max(max_dist, 1e-6)
                cost_matrix[i, j] = dist_cost + config.track_area_log_weight * area_cost

    finite_mask = np.isfinite(cost_matrix)
    valid_rows  = np.where(finite_mask.any(axis=1))[0]
    valid_cols  = np.where(finite_mask.any(axis=0))[0]
    if len(valid_rows) == 0 or len(valid_cols) == 0:
        return []

    large_penalty = 1e9
    reduced = np.where(np.isfinite(cost_matrix[np.ix_(valid_rows, valid_cols)]),
                       cost_matrix[np.ix_(valid_rows, valid_cols)], large_penalty)

    bad_rows = np.all(reduced >= large_penalty, axis=1)
    bad_cols = np.all(reduced >= large_penalty, axis=0)
    if bad_rows.any() or bad_cols.any():
        keep_r = np.where(~bad_rows)[0]
        keep_c = np.where(~bad_cols)[0]
        if len(keep_r) == 0 or len(keep_c) == 0:
            return []
        reduced    = reduced   [np.ix_(keep_r, keep_c)]
        valid_rows = valid_rows[keep_r]
        valid_cols = valid_cols[keep_c]

    row_ind, col_ind = linear_sum_assignment(reduced)

    matches = []
    for r, c in zip(row_ind, col_ind):
        orig_i = valid_rows[r]
        orig_j = valid_cols[c]
        link_cost = cost_matrix[orig_i, orig_j]
        if not np.isfinite(link_cost):
            continue
        pr = prev_cluster.iloc[orig_i]
        cr = curr_cluster.iloc[orig_j]
        dist_px = math.hypot(
            float(cr["centroid_x_px"]) - float(pr["centroid_x_px"]),
            float(cr["centroid_y_px"]) - float(pr["centroid_y_px"]),
        )
        matches.append({
            "prev_index":        int(pr["global_index"]),
            "curr_index":        int(cr["global_index"]),
            "link_cost":         float(link_cost),
            "link_distance_px":  float(dist_px),
            "dbscan_cluster_id":   int(pr.get("dbscan_cluster_id", -1)),
            "dbscan_cluster_size": int(pr.get("dbscan_cluster_size", 0)),
            "dbscan_is_crowded":   bool(pr.get("dbscan_is_crowded", False)),
        })
    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy().reset_index(drop=True)
    df["global_index"] = np.arange(len(df), dtype=int)
    df = df.sort_values(["t", "nucleus_3d_id"]).reset_index(drop=True)

    df["track_id"]                = -1
    df["track_link_distance_px"]  = np.nan
    df["track_link_cost"]         = np.nan
    df["track_dbscan_cluster_id"] = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"] = False
    df["track_link_method"]       = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    for idx in df.index[df["t"] == time_values[0]]:
        df.at[idx, "track_id"]          = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(
            prev_df, curr_df, config)

        all_matches: List[dict] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            pc = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            cc = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].reset_index(drop=True)
            all_matches.extend(match_cluster_with_assignment(pc, cc, config))

        matched_curr_global: set = set()

        # FIX: loop body was previously unreachable due to a misindented `continue`.
        for match in sorted(all_matches, key=lambda x: x["link_cost"]):
            prev_global = int(match["prev_index"])
            curr_global = int(match["curr_index"])

            if curr_global in matched_curr_global:
                continue

            track_id = int(df.at[prev_global, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global, "track_id"]          = track_id
                df.at[prev_global, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global)
            df.at[curr_global, "track_id"]                  = track_id
            df.at[curr_global, "track_link_distance_px"]    = float(match["link_distance_px"])
            df.at[curr_global, "track_link_cost"]           = float(match["link_cost"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(match["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(match["dbscan_is_crowded"])
            df.at[curr_global, "track_link_method"]         = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global = int(row["index"])
            df.at[curr_global, "track_dbscan_cluster_id"]   = int(row["dbscan_cluster_id"])
            df.at[curr_global, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global, "track_dbscan_is_crowded"]   = bool(row["dbscan_is_crowded"])
            if df.at[curr_global, "track_id"] == -1:
                df.at[curr_global, "track_id"]          = next_track_id
                df.at[curr_global, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()
    track_len = tracked_df.groupby("track_id").size()
    return pd.DataFrame([{
        "n_rows":                int(len(tracked_df)),
        "n_tracks":              int(tracked_df["track_id"].nunique()),
        "median_track_length":   float(track_len.median()),
        "mean_track_length":     float(track_len.mean()),
        "fraction_crowded_links":float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks":   float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }])


## 18. Fiji-equivalent cumulative halos

In [ ]:
def build_cumulative_halo_masks(nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, np.ndarray]:
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    masks = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        masks[f"halo_{i}_cum"] = dist_outside <= i * config.halo_step_px

    for i in range(1, config.n_halos + 1):
        prev = nucleus_mask if i == 1 else masks[f"halo_{i-1}_cum"]
        masks[f"ring_{i}"] = masks[f"halo_{i}_cum"] & ~prev

    masks["nucleus_eroded"] = (
        ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        if config.erosion_px > 0 else nucleus_mask.copy()
    )
    masks["cytoplasm_mask"] = masks[f"halo_{config.n_halos}_cum"] & ~nucleus_mask
    return masks


def _mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {"area_px": area_px, "intden": float(vals.sum()), "mean_intensity": float(vals.mean())}


def measure_fiji_equivalent_halos(
    intensity_image: np.ndarray,
    nucleus_mask: np.ndarray,
    config: PipelineConfig,
) -> Dict[str, float]:
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out: Dict[str, float] = {}

    for key in ("nucleus_mask", "nucleus_eroded", "cytoplasm_mask"):
        prefix = key.replace("_mask", "").replace("_cum", "")
        for k, v in _mask_stats(intensity_image, masks[key]).items():
            out[f"{prefix}_{k}"] = v

    for i in range(1, config.n_halos + 1):
        for k, v in _mask_stats(intensity_image, masks[f"halo_{i}_cum"]).items():
            out[f"halo_{i}_cum_{k}"] = v
        for k, v in _mask_stats(intensity_image, masks[f"ring_{i}"]).items():
            out[f"ring_{i}_{k}"] = v

    nm = out.get("nucleus_mean_intensity", np.nan)
    cm = out.get("cytoplasm_mean_intensity", np.nan)
    out["nc_ratio"]          = float(nm / cm)          if np.isfinite(nm) and np.isfinite(cm) and cm != 0         else np.nan
    out["nc_ratio_fraction"] = float(nm / (nm + cm))   if np.isfinite(nm) and np.isfinite(cm) and (nm + cm) != 0  else np.nan
    return out


def recover_nucleus_mask_from_plane(
    t_idx: int, z_idx: int,
    centroid_x_px: float, centroid_y_px: float,
    config: PipelineConfig,
) -> np.ndarray:
    mask_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
    mask = np.load(mask_path).astype(bool)
    labeled = measure.label(mask, connectivity=2)
    best_label, best_dist = None, np.inf
    for prop in measure.regionprops(labeled):
        cy, cx = prop.centroid
        dist = math.hypot(cx - centroid_x_px, cy - centroid_y_px)
        if dist < best_dist:
            best_dist, best_label = dist, prop.label
    return (labeled == best_label) if best_label is not None else np.zeros_like(mask, dtype=bool)


def _process_single_tracked_nucleus(row, img_5d: np.ndarray, config: PipelineConfig) -> dict:
    nucleus_mask  = recover_nucleus_mask_from_plane(
        int(row.t), int(row.z), float(row.centroid_x_px), float(row.centroid_y_px), config)
    nuclear_plane = get_nuclear_plane(img_5d, int(row.t), int(row.z), config)
    halo_metrics  = measure_fiji_equivalent_halos(nuclear_plane, nucleus_mask, config)
    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"]   = rec.get("nucleus_area_px", 0) * config.pixel_size_um ** 2
    rec["cytoplasm_area_um2"] = rec.get("cytoplasm_area_px", 0) * config.pixel_size_um ** 2
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df
    rows = Parallel(n_jobs=max(1, int(config.n_workers)), backend="threading", batch_size=1)(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )
    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df


## 19. Radial sweep placeholder

In [ ]:
def radial_sweep_membrane_placeholder(
    membrane_plane: np.ndarray,
    nucleus_mask: np.ndarray,
    centroid_x_px: float,
    centroid_y_px: float,
    n_angles: int = 180,
    max_radius_px: int = 200,
) -> pd.DataFrame:
    angles = np.linspace(0, 360, n_angles, endpoint=False)
    return pd.DataFrame({
        "angle_deg":     angles.astype(float),
        "peak_intensity": np.full(n_angles, np.nan),
        "peak_radius_px": np.full(n_angles, np.nan),
    })


## 20. Portable export helper

In [ ]:
def _save_portable_table(df: pd.DataFrame, out_stem: Path) -> dict:
    """Save a table as CSV (human-readable) + pickle (lossless, fast).
    Parquet removed — requires pyarrow/fastparquet which are not available."""
    out_stem.parent.mkdir(parents=True, exist_ok=True)
    saved: dict = {}
    if df is None:
        return saved
    csv_path = out_stem.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    saved["csv"] = str(csv_path)
    pkl_path = out_stem.with_suffix(".pkl")
    df.to_pickle(pkl_path)
    saved["pickle"] = str(pkl_path)
    return saved


def export_portable_analysis_bundle(
    config: PipelineConfig,
    export_dir: Union[str, Path],
    seg_index_df: Optional[pd.DataFrame] = None,
    objects_df: Optional[pd.DataFrame] = None,
    grouped_z_df: Optional[pd.DataFrame] = None,
    best_z_df: Optional[pd.DataFrame] = None,
    filtered_df: Optional[pd.DataFrame] = None,
    timed_df: Optional[pd.DataFrame] = None,
    tracked_df: Optional[pd.DataFrame] = None,
    halo_df: Optional[pd.DataFrame] = None,
    copy_mask_npy_files: bool = False,
) -> Path:
    export_dir = Path(export_dir)
    dirs = {k: export_dir / k for k in ("tables", "masks", "config", "notes")}
    for p in [export_dir] + list(dirs.values()):
        p.mkdir(parents=True, exist_ok=True)

    manifest = {
        "created_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "input_image_name": config.input_image_name,
        "input_image_path_on_cluster": str(config.input_image_path),
        "pixel_size_um": float(config.pixel_size_um),
        "z_step_um": float(config.z_step_um),
        "nuclear_channel_index": int(config.nuclear_channel_index),
        "membrane_channel_index": int(config.membrane_channel_index),
        "copied_files": {}, "tables": {},
    }

    config_path = dirs["config"] / "pipeline_config.json"
    config_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))
    manifest["copied_files"]["pipeline_config"] = str(config_path)

    for key, src in {
        "segmentation_class_hyperstack":       config.segmentation_class_hyperstack_path,
        "segmentation_probability_hyperstack": config.segmentation_probability_hyperstack_path,
        "segmentation_label_hyperstack":       config.segmentation_label_hyperstack_path,
        "nucleus_instance_hyperstack":         config.nucleus_instance_hyperstack_path,
        "droplet_instance_hyperstack":         config.droplet_instance_hyperstack_path,
    }.items():
        src = Path(src)
        if src.exists():
            dst = dirs["masks"] / src.name
            shutil.copy2(src, dst)
            manifest["copied_files"][key] = str(dst)

    for name, df in {
        "segmentation_index":           seg_index_df,
        "plane_objects":                objects_df,
        "grouped_z_objects":            grouped_z_df,
        "best_z_nuclei":                best_z_df,
        "best_z_nuclei_with_exclusion": filtered_df,
        "best_z_nuclei_timed":          timed_df,
        "tracked_nuclei":               tracked_df,
        "halo_analysis":                halo_df,
    }.items():
        if df is not None:
            manifest["tables"][name] = _save_portable_table(df, dirs["tables"] / name)

    if copy_mask_npy_files and seg_index_df is not None and "mask_path" in seg_index_df.columns:
        plane_mask_dir = dirs["masks"] / "plane_npy_masks"
        plane_mask_dir.mkdir(parents=True, exist_ok=True)
        portable_idx = seg_index_df.copy()
        new_paths = []
        for row in portable_idx.itertuples(index=False):
            src = Path(row.mask_path)
            if src.exists():
                dst = plane_mask_dir / src.name
                shutil.copy2(src, dst)
                new_paths.append(str(dst))
            else:
                new_paths.append(None)
        portable_idx["portable_mask_path"] = new_paths
        manifest["tables"]["segmentation_index_portable"] = _save_portable_table(
            portable_idx, dirs["tables"] / "segmentation_index_portable")

    readme = (
        "Portable laptop analysis bundle\n\n"
        "Contents\n--------\n"
        "masks/  — Hyperstack TIFFs for segmentation QC, instance labels, and napari overlays.\n"
        "tables/ — Analysis tables (CSV + pickle).\n"
        "config/ — Pipeline configuration snapshot.\n\n"
        "Recommended workflow\n--------------------\n"
        "1. Open TIFF files in napari for segmentation debugging.\n"
        "2. Load halo_analysis.pkl or .csv for plotting and filtering.\n"
        "3. Use tracked_nuclei / best_z_nuclei tables for re-analysis without rerunning the model.\n"
    )
    (dirs["notes"] / "README.txt").write_text(readme)
    (export_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return export_dir


## 21. QC plotting helpers

In [ ]:
def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config: PipelineConfig,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Overlay raw nuclear channel with the binary nucleus mask."""
    T, Z, C, Y, X = img_5d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, t_idx in zip(axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        raw = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        if raw.max() > 0:
            raw /= raw.max()
        ax.imshow(raw, cmap="gray")
        ax.imshow(nucleus_mask_4d[t_idx, z_idx].astype(bool), alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")

    for ax in axes[len(timepoints):]:
        ax.axis("off")

    fig.suptitle("QC: Nuclear channel + nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4,
) -> None:
    """Mask snapshots with contours + total nuclear area over time."""
    from skimage.measure import find_contours
    T, Z, Y, X = nucleus_mask_4d.shape
    if timepoints is None:
        timepoints = np.linspace(0, T - 1, min(T, 6), dtype=int).tolist()

    n_cols = min(max_cols, len(timepoints))
    n_rows = math.ceil(len(timepoints) / n_cols)
    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs  = fig.add_gridspec(n_rows + 1, n_cols)

    panel_axes = [fig.add_subplot(gs[r, c]) for r in range(n_rows) for c in range(n_cols)]

    for ax, t_idx in zip(panel_axes, timepoints):
        z_idx = (Z // 2 if z_mode == "middle"
                 else int(np.argmax(nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1))))
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled    = measure.label(mask_plane)
        ax.imshow(mask_plane, cmap="gray")
        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)
        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")

    for ax in panel_axes[len(timepoints):]:
        ax.axis("off")

    area_ax = fig.add_subplot(gs[n_rows, :])
    area_ax.plot(np.arange(T), nucleus_mask_4d.reshape(T, -1).sum(axis=1), marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")
    plt.tight_layout()
    plt.show()


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """Count connected nucleus objects across all Z slices per time point."""
    T, Z = nucleus_mask_4d.shape[:2]
    counts = [
        sum(measure.label(nucleus_mask_4d[t, z].astype(bool)).max() for z in range(Z))
        for t in range(T)
    ]
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(np.arange(T), counts, marker="o")
    ax.set_xlabel("Time index")
    ax.set_ylabel("Connected nucleus objects")
    ax.set_title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()


def plot_best_focus_nuclear_planes(
    img_5d: np.ndarray,
    config: PipelineConfig,
    figsize_per_panel: float = 4.0,
) -> pd.DataFrame:
    """Plot the single best-focus nuclear-channel Z plane for each time point."""
    n_t = img_5d.shape[0]
    n_cols = min(5, n_t)
    n_rows = math.ceil(n_t / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(figsize_per_panel * n_cols, figsize_per_panel * n_rows))
    axes = np.array(axes).reshape(-1)
    focus_rows = []

    for t in range(n_t):
        keep_z, scores, best_z = get_best_focus_z_indices(
            img_5d, t,
            nuc_channel_idx=config.effective_focus_channel_index,
            exclude_edge_z=config.focus_edge_z_exclusion,
            window_radius=0,
            metric=config.focus_metric,
            pixel_size_um=config.pixel_size_um,
            config=config,
        )
        ax = axes[t]
        if best_z is None:
            ax.set_title(f"t={t}: no valid focus plane")
            ax.axis("off")
            focus_rows.append({"t": t, "best_z": np.nan, "focus_score": np.nan,
                                "focus_metric": config.focus_metric,
                                "nuclear_channel_index": config.effective_focus_channel_index})
            continue
        nuc2d      = img_5d[t, best_z, config.effective_focus_channel_index]
        best_score = float(scores[best_z])
        ax.imshow(nuc2d, cmap="gray")
        ax.set_title(f"t={t}, best z={best_z}\nfocus={best_score:.4g}")
        ax.axis("off")
        focus_rows.append({"t": t, "best_z": int(best_z), "focus_score": best_score,
                           "focus_metric": config.focus_metric,
                           "nuclear_channel_index": config.effective_focus_channel_index})

    for ax in axes[n_t:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(focus_rows)


def plot_nucleus_probability_qc(class_prob_map: np.ndarray, raw_plane: np.ndarray,
                                title: str = "") -> None:
    """Show raw plane, nucleus probability map, and its histogram."""
    nucleus_prob = class_prob_map[..., 2]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(raw_plane, cmap="gray"); axes[0].set_title("Raw"); axes[0].axis("off")
    im = axes[1].imshow(nucleus_prob, cmap="inferno")
    axes[1].set_title("Nucleus probability"); plt.colorbar(im, ax=axes[1])
    axes[2].hist(nucleus_prob.ravel(), bins=100); axes[2].set_title("Probability distribution")
    fig.suptitle(title); plt.tight_layout(); plt.show()


def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("N / (N + C)")
    ax.set_title("N/C ratio fraction by track")
    plt.tight_layout(); plt.show()


def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot."); return
    fig, ax = plt.subplots(figsize=(7, 4))
    for track_id, df_t in halo_df.groupby("track_id"):
        df_t = df_t.sort_values("true_time_min")
        ax.plot(df_t["true_time_min"], df_t["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear area (µm²)")
    ax.set_title("Largest nuclear cross-sectional area by track")
    plt.tight_layout(); plt.show()


def plot_largest_cross_sectional_area_vs_time(
    df: pd.DataFrame,
    config: PipelineConfig,
) -> None:
    """
    Scatter all individual nuclear areas plus population mean vs true acquisition time.

    Works with timed_df, tracked_df, or halo_df.  Requires 'true_time_min'.
    """
    if df.empty:
        print("No data to plot."); return
    plot_df = df.copy()
    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("DataFrame must contain 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * config.pixel_size_um ** 2
    if "true_time_min" not in plot_df.columns:
        raise ValueError("DataFrame must contain 'true_time_min'.")

    plot_df = plot_df.sort_values("true_time_min")
    mean_df = (plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
               .mean().rename(columns={"nucleus_area_um2": "mean_area"}))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["true_time_min"], plot_df["nucleus_area_um2"], alpha=0.35, s=20,
               label="Individual nuclei")
    ax.plot(mean_df["true_time_min"], mean_df["mean_area"], linewidth=2, label="Population mean")
    ax.set_xlabel("True acquisition time (min)")
    ax.set_ylabel("Nuclear cross-sectional area (µm²)")
    ax.set_title("Nuclear area vs time")
    ax.legend(); plt.tight_layout(); plt.show()


## 22. Stage execution flags

Set a flag to `True` to recompute that stage; `False` loads the saved artefact from disk.

In [ ]:
RUN_SEGMENTATION      = True
RUN_OBJECT_EXTRACTION = True
RUN_GROUP_ACROSS_Z    = True
RUN_SELECT_BEST_Z     = True
RUN_FLAG_CLOSE_NUCLEI = True
RUN_ADD_TIMING        = True
RUN_TRACKING          = True
RUN_HALO_ANALYSIS     = True
RUN_EXPORT_PORTABLE   = True

DEBUG_T_IDX = 0
DEBUG_Z_IDX = 0


## 23. Initialise runtime and confirm paths

In [ ]:
configure_tensorflow_for_gpu(cfg)

print(f"Model path:      {cfg.model_path}")
print(f"Input image:     {cfg.input_image_path}")
print(f"Output dir:      {cfg.output_dir}")

assert cfg.model_path.exists(),       f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "debug_pipeline_config.json")
print("Configuration saved.")


## 24. Load image

In [ ]:
t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Shape: {img_5d.shape}  |  dtype: {img_5d.dtype}  |  load time: {time.perf_counter() - t0:.2f}s")


## 25. Optional focus-score QC

In [ ]:
# Quick sanity-check for one time point.
keep_z, scores, best_z = get_best_focus_z_indices(
    img_5d=img_5d, t=DEBUG_T_IDX,
    nuc_channel_idx=cfg.effective_focus_channel_index,
    exclude_edge_z=cfg.focus_edge_z_exclusion,
    window_radius=cfg.focus_window_radius,
    metric=cfg.focus_metric,
    pixel_size_um=cfg.pixel_size_um,
    config=cfg,
)
print(f"Best Z: {best_z}  |  Keep Z: {keep_z}")
print(f"Scores: {scores}")

# Uncomment to plot all time points:
# focus_qc_df = plot_best_focus_nuclear_planes(img_5d=img_5d, config=cfg)
# display(focus_qc_df)


## 26. Single-plane segmentation debug

In [ ]:
# Load the model once here; the full-segmentation cell (Section 27) reuses it.
model = None

if RUN_SEGMENTATION:
    model = load_unet_model(cfg.model_path)
    print("Model loaded.")

plane_yxc = get_full_plane_yxc(img_5d, DEBUG_T_IDX, DEBUG_Z_IDX)
print(f"Debug plane shape: {plane_yxc.shape}")

if model is not None:
    cfg.batch_size = 1  # memory-safe inference
    debug_prob_map, debug_class_map, debug_nucleus_mask, debug_instance_labels = (
        segment_single_plane_with_overlap(
            plane_yxc, model, cfg,
            patch_size=cfg.patch_size, stride=cfg.patch_stride,
        )
    )
    print("Single-plane debug segmentation complete.")
    print(f"  Probability map shape: {debug_prob_map.shape}")
    print(f"  Unique class labels:   {np.unique(debug_class_map)}")
    print(f"  Nucleus pixels:        {int(debug_nucleus_mask.sum())}")
    gc.collect()


## 27. Run or load full segmentation

### ⚠️ Before running this cell

If you have edited any function cells above (Sections 6–10), you **must restart the kernel and re-run all cells from the top** before running segmentation.

The kernel caches function definitions in memory. Running only the edited cell updates the source but does **not** replace the compiled function object the segmentation loop calls. A stale cached definition is the cause of `NameError: class_mask_5d` and similar errors even when the notebook source looks correct.

**Restart kernel → Run All** (or Kernel → Restart & Run All Cells).

In [ ]:
if RUN_SEGMENTATION:
    if model is None:   # might have been skipped in Section 26
        model = load_unet_model(cfg.model_path)
    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(img_5d=img_5d, model=model, config=cfg)
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f}s")
else:
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Loaded saved segmentation index.")

print(seg_index_df.shape)
display(seg_index_df.head())


## 28. Run or load object extraction

In [ ]:
objects_path = cfg.obj_dir / "plane_objects.pkl"

if RUN_OBJECT_EXTRACTION:
    t0 = time.perf_counter()
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
    print(f"Object extraction finished in {time.perf_counter() - t0:.2f}s")
else:
    objects_df = pd.read_pickle(objects_path)
    print("Loaded saved object table.")

print(objects_df.shape)
display(objects_df.head())


## 29. Run or load Z grouping

In [ ]:
grouped_z_path = cfg.obj_dir / "grouped_z_objects.pkl"

if RUN_GROUP_ACROSS_Z:
    t0 = time.perf_counter()
    grouped_z_df = group_nuclei_across_z(objects_df, cfg)
    print(f"Z grouping finished in {time.perf_counter() - t0:.2f}s")
else:
    grouped_z_df = pd.read_pickle(grouped_z_path)
    print("Loaded grouped-Z table.")

print(grouped_z_df.shape)
display(grouped_z_df.head())


## 30. Run or load best-Z selection

In [ ]:
best_z_path = cfg.obj_dir / "best_z_nuclei.pkl"

if RUN_SELECT_BEST_Z:
    t0 = time.perf_counter()
    best_z_df = select_best_z_per_nucleus(grouped_z_df, cfg)
    print(f"Best-Z selection finished in {time.perf_counter() - t0:.2f}s")
else:
    best_z_df = pd.read_pickle(best_z_path)
    print("Loaded best-Z table.")

print(best_z_df.shape)
display(best_z_df.head())


## 31. Optional area-based cleanup

In [ ]:
def filter_nuclei_by_area(
    df: pd.DataFrame,
    config: PipelineConfig,
    min_area_um2: float = 100.0,
    max_area_um2: float = 300.0,
) -> pd.DataFrame:
    """Remove objects that are too small or too large to be real nuclei."""
    df = df.copy()
    if "nucleus_area_um2" not in df.columns:
        df["nucleus_area_um2"] = df["area_px"] * config.pixel_size_um ** 2
    filtered = df[(df["nucleus_area_um2"] >= min_area_um2) & (df["nucleus_area_um2"] <= max_area_um2)]
    print(f"Filtered: {len(df)} → {len(filtered)} nuclei")
    return filtered


# Optional debug usage:
# best_z_area_filtered_df = filter_nuclei_by_area(best_z_df, cfg, 100, 300)
# display(best_z_area_filtered_df.head())


## 32. Run or load close-nuclei exclusion

In [ ]:
filtered_path = cfg.obj_dir / "best_z_nuclei_with_exclusion.pkl"

if RUN_FLAG_CLOSE_NUCLEI:
    t0 = time.perf_counter()
    filtered_df = flag_close_nuclei(best_z_df, cfg)
    print(f"Close-nuclei flagging finished in {time.perf_counter() - t0:.2f}s")
else:
    filtered_df = pd.read_pickle(filtered_path)
    print("Loaded exclusion table.")

print(filtered_df.shape)
display(filtered_df.head())


## 33. Run or load timing metadata

In [ ]:
timed_path = cfg.track_dir / "best_z_nuclei_timed.pkl"

if RUN_ADD_TIMING:
    t0 = time.perf_counter()
    timed_df = add_tile_timing_metadata(filtered_df, img_5d, cfg)
    print(f"Timing metadata finished in {time.perf_counter() - t0:.2f}s")
else:
    timed_df = pd.read_pickle(timed_path)
    print("Loaded timed nuclei table.")

print(timed_df.shape)
display(timed_df.head())


## 34. Run or load tracking

In [ ]:
tracked_path = cfg.track_dir / "tracked_nuclei.pkl"

if RUN_TRACKING:
    t0 = time.perf_counter()
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg, valid_only=False)
    print(f"Hybrid DBSCAN tracking finished in {time.perf_counter() - t0:.2f}s")
else:
    tracked_df = pd.read_pickle(tracked_path)
    print("Loaded tracked nuclei table.")

print(tracked_df.shape)
display(tracked_df.head())
display(summarize_tracking_debug(tracked_df))


## 35. Run or load halo analysis

In [ ]:
halo_path = cfg.analysis_dir / "halo_analysis.pkl"

if RUN_HALO_ANALYSIS:
    t0 = time.perf_counter()
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Halo analysis finished in {time.perf_counter() - t0:.2f}s")
else:
    halo_df = pd.read_pickle(halo_path)
    print("Loaded halo analysis table.")

print(halo_df.shape)
display(halo_df.head())


## 36. Load saved mask hyperstacks for QC

In [ ]:
class_mask_5d       = tiff.imread(cfg.segmentation_class_hyperstack_path)
label_mask_4d       = tiff.imread(cfg.segmentation_label_hyperstack_path)
nucleus_instance_4d = tiff.imread(cfg.nucleus_instance_hyperstack_path)

print(f"class_mask_5d shape:       {class_mask_5d.shape}")
print(f"label_mask_4d shape:       {label_mask_4d.shape}")
print(f"nucleus_instance_4d shape: {nucleus_instance_4d.shape}")

nucleus_mask_4d = class_mask_5d[:, :, cfg.nucleus_class_index] > 0
print(f"nucleus_mask_4d shape:     {nucleus_mask_4d.shape}")


## 37. QC plots

In [ ]:
plot_nucleus_mask_overlay_qc(img_5d, nucleus_mask_4d, cfg, z_mode="max_mask")
plot_nucleus_mask_qc_summary(nucleus_mask_4d, z_mode="max_mask")
plot_nucleus_count_over_time(nucleus_mask_4d)


## 38. Downstream measurement plots

In [ ]:
plot_nc_vs_time(halo_df)
plot_area_vs_time(halo_df)
plot_largest_cross_sectional_area_vs_time(timed_df, cfg)


## 39. Probability-map debug on the selected plane

In [ ]:
if "debug_prob_map" in dir():
    raw_plane = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.nuclear_channel_index]
    plot_nucleus_probability_qc(debug_prob_map, raw_plane,
                                title=f"t={DEBUG_T_IDX}, z={DEBUG_Z_IDX}")
else:
    print("Run Section 26 (single-plane debug) first.")


## 40. Pipeline summary

In [ ]:
summary_df = pd.DataFrame([{
    "seg_index_rows": len(seg_index_df),
    "objects_rows":   len(objects_df),
    "grouped_z_rows": len(grouped_z_df),
    "best_z_rows":    len(best_z_df),
    "filtered_rows":  len(filtered_df),
    "timed_rows":     len(timed_df),
    "tracked_rows":   len(tracked_df),
    "halo_rows":      len(halo_df),
}])
display(summary_df)


## 41. Export portable analysis bundle

In [ ]:
if RUN_EXPORT_PORTABLE:
    portable_dir = cfg.output_dir / "portable_laptop_bundle"
    exported_path = export_portable_analysis_bundle(
        config=cfg,
        export_dir=portable_dir,
        seg_index_df=seg_index_df,
        objects_df=objects_df,
        grouped_z_df=grouped_z_df,
        best_z_df=best_z_df,
        filtered_df=filtered_df,
        timed_df=timed_df,
        tracked_df=tracked_df,
        halo_df=halo_df,
        copy_mask_npy_files=False,
    )
    print(f"Portable bundle written to: {exported_path}")


## 42. (Optional) Inspect probability TIFF

In [ ]:
prob_tiff_path = cfg.segmentation_probability_hyperstack_path
if prob_tiff_path.exists():
    prob_hyperstack = tiff.imread(prob_tiff_path)
    print(f"Shape: {prob_hyperstack.shape}  |  dtype: {prob_hyperstack.dtype}")
    print(f"Range: {float(np.nanmin(prob_hyperstack)):.4f} – {float(np.nanmax(prob_hyperstack)):.4f}")
else:
    print(f"Probability TIFF not found yet: {prob_tiff_path}")


## 43. Notes

- Every major stage is exposed as its own runnable block — there is no monolithic wrapper.
- Intermediate tables are written to disk so you can rerun only the stage being debugged.
- Set `RUN_EXPORT_PORTABLE = True` before the final cell to bundle TIFFs + CSVs for local laptop analysis.
